In [1]:

import numpy as np
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# --------------------------
# Simulación de datos
# --------------------------
np.random.seed(42)
n_assets = 8
n_obs = 250
returns = np.random.normal(0.0005, 0.01, size=(n_obs, n_assets))
matriz_varianza = np.cov(returns.T)
mu = returns.mean(axis=0)

# Benchmark (por ejemplo, un índice de referencia)
w_b = np.random.rand(n_assets)
w_b /= w_b.sum()

# Portafolio actual (situación actual del fondo)
w_0 = w_b + np.random.uniform(-0.02, 0.02, size=n_assets)
w_0 = np.clip(w_0, 0, None)
w_0 /= w_0.sum()

# Parámetros del modelo
tau_max = 0.50     # 10% máximo turnover
w_min = np.zeros(n_assets)
w_max = np.ones(n_assets) * 0.4  # máximo 40% por activo

# Creación del modelo Gurobi
m = gp.Model("Optimizacion_de_varianza_minima")

# Variables de decisión:
# w_i: pesos del portafolio
# u_i: variables auxiliares para modelar |w_i - w_0_i|
w = m.addMVar(shape=n_assets, lb=w_min, ub=w_max, name="w")
u = m.addMVar(shape=n_assets, lb=0.0, name="u")

# Restricciones
m.addConstr(w.sum() == 1, name="Totalmente_invertido")
m.addConstrs((w[i] >= 0 for i in range(n_assets)), name="no_short_selling")
m.addConstrs((u[i] >= w[i] - w_0[i] for i in range(n_assets)), name="turnover_pos")
m.addConstrs((u[i] >= -(w[i] - w_0[i]) for i in range(n_assets)), name="turnover_neg")
m.addConstr(u.sum() <= tau_max, name="turnover_bound")

# Función objetivo: minimizar Tracking Error Varianza
# --------------------------
# (w - w_b)^T * Σ * (w - w_b)
tracking_error = (w - w_b) @ matriz_varianza @ (w - w_b)
m.setObjective(tracking_error, GRB.MINIMIZE)

# Resolver el modelo
m.Params.OutputFlag = 1   # mostrar el log de optimización
m.optimize()

# Resultados
if m.status == GRB.OPTIMAL:
    w_opt = w.X
    turnover = np.sum(np.abs(w_opt - w_0))  
    results = pd.DataFrame({
        "Asset": [f"Asset {i+1}" for i in range(n_assets)],
        "w_current": w_0,
        "w_benchmark": w_b,
        "w_optimal": w_opt
    })
    
    print("\n Resultados del portafolio optimizado")
    print(results.round(4))
    print("\nTracking Error Varianza:",
          np.dot((w_opt - w_b).T, matriz_varianza @ (w_opt - w_b)))
    print("Turnover realizado:", turnover)
else:
    print("El modelo no encontró una solución óptima.")

Set parameter Username
Set parameter LicenseID to value 2613029
Academic license - for non-commercial use only - expires 2026-01-22
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-1335U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 26 rows, 16 columns and 56 nonzeros
Model fingerprint: 0xdd52965b
Model has 36 quadratic objective terms
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [9e-07, 5e-05]
  QObjective range [2e-08, 2e-04]
  Bounds range     [4e-01, 4e-01]
  RHS range        [2e-03, 1e+00]
Presolve removed 8 rows and 0 columns
Presolve time: 0.03s
Presolved: 18 rows, 16 columns, 48 nonzeros
Presolved model has 36 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 Free vars  : 7
 AA' NZ     : 1.380e+02
 Factor NZ  : 3.250e+02
 Factor Op

In [2]:
w_b

array([0.10954648, 0.0177623 , 0.09386257, 0.02986799, 0.21748444,
       0.25500918, 0.01945932, 0.25700772])

In [3]:
w_0

array([0.11493496, 0.01014067, 0.08006151, 0.03110693, 0.23857563,
       0.27398509, 0.00152799, 0.24966723])

In [3]:
matriz_varianza

array([[ 8.22513054e-05,  1.28017976e-06, -6.52051965e-07,
         6.99802624e-08, -5.10401006e-06, -4.28183883e-06,
        -2.39560056e-06, -1.03581708e-05],
       [ 1.28017976e-06,  9.54594622e-05, -3.96444990e-06,
         1.95679256e-06,  7.44957240e-06, -9.45356433e-06,
         2.75822212e-06,  7.44234860e-06],
       [-6.52051965e-07, -3.96444990e-06,  8.97863549e-05,
        -1.22545513e-06,  3.36197841e-06, -7.45364979e-06,
        -2.85726895e-07, -7.50345434e-07],
       [ 6.99802624e-08,  1.95679256e-06, -1.22545513e-06,
         1.01733765e-04, -1.35610465e-05, -3.68256270e-07,
         8.81996269e-06,  1.29754900e-06],
       [-5.10401006e-06,  7.44957240e-06,  3.36197841e-06,
        -1.35610465e-05,  8.91534942e-05, -5.85463909e-06,
        -9.46376026e-06,  3.24783203e-06],
       [-4.28183883e-06, -9.45356433e-06, -7.45364979e-06,
        -3.68256270e-07, -5.85463909e-06,  1.08807441e-04,
        -4.13067993e-09,  4.04853656e-07],
       [-2.39560056e-06,  2.758222

In [5]:
w_opt

array([0.10954723, 0.01776145, 0.09386167, 0.02986878, 0.21748539,
       0.2550097 , 0.01945869, 0.2570071 ])